# Hybrid Ink Strategy: Switching Between Ink v14 and Ink v5

This notebook implements a hybrid trading strategy that switches between Ink v14 and Ink v5 strategies based on strong buy/sell signals from Ink v14.

## Import Required Libraries

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# For strategy simulation
from datetime import datetime, timedelta
import random

# Set plotting style
plt.style.use('ggplot')
sns.set_theme()

## Load Ink v14 and Ink v5 Strategies

We'll define simplified versions of Ink v14 and Ink v5 strategies for demonstration purposes.
In a real implementation, you would load your actual strategy implementations or models.

In [ ]:
# Define simplified Ink v14 strategy (returns signal and signal strength)
def ink_v14_strategy(prices, volumes, window=14):
    """
    Simulated Ink v14 strategy that generates trading signals and signal strength
    
    Parameters:
    - prices: Series or list of historical prices
    - volumes: Series or list of historical volumes
    - window: Lookback period
    
    Returns:
    - signal: 1 for buy, -1 for sell, 0 for neutral
    - strength: A value between 0 and 1 indicating signal strength
    """
    if len(prices) < window:
        return 0, 0.0
    
    # Convert to pandas Series if they're not already
    prices = pd.Series(prices) if not isinstance(prices, pd.Series) else prices
    volumes = pd.Series(volumes) if not isinstance(volumes, pd.Series) else volumes
    
    # Calculate some indicators (simplified for demonstration)
    price_change = prices.pct_change(window).iloc[-1]
    volume_change = volumes.pct_change(window).iloc[-1]
    momentum = prices.diff(window).iloc[-1]
    
    # Determine signal
    if price_change > 0 and volume_change > 0 and momentum > 0:
        signal = 1  # Buy signal
    elif price_change < 0 and momentum < 0:
        signal = -1  # Sell signal
    else:
        signal = 0  # Neutral
    
    # Calculate strength (simplified)
    strength = abs(price_change * volume_change) if signal != 0 else 0.0
    strength = min(1.0, strength * 10)  # Scale to [0,1]
    
    return signal, strength

# Define simplified Ink v5 strategy (returns only signal)
def ink_v5_strategy(prices, window=5):
    """
    Simulated Ink v5 strategy that generates trading signals
    
    Parameters:
    - prices: Series or list of historical prices
    - window: Lookback period
    
    Returns:
    - signal: 1 for buy, -1 for sell, 0 for neutral
    """
    if len(prices) < window:
        return 0
    
    # Convert to pandas Series if not already
    prices = pd.Series(prices) if not isinstance(prices, pd.Series) else prices
    
    # Simple moving average crossover strategy
    short_ma = prices.rolling(window=window).mean().iloc[-1]
    long_ma = prices.rolling(window=window*2).mean().iloc[-1]
    
    if short_ma > long_ma:
        return 1  # Buy signal
    elif short_ma < long_ma:
        return -1  # Sell signal
    else:
        return 0  # Neutral

## Define Signal Strength Threshold

We'll set a threshold to determine what constitutes a strong buy/sell signal from the Ink v14 strategy.
When the signal strength exceeds this threshold, we'll use the Ink v14 strategy; otherwise, we'll use Ink v5.

In [ ]:
# Define the threshold for strong signals (adjust this based on your strategy characteristics)
SIGNAL_STRENGTH_THRESHOLD = 0.7  # Signal strength must be at least 0.7 to be considered strong

def should_use_ink_v14(signal_strength):
    """
    Determines whether to use Ink v14 or Ink v5 based on signal strength
    
    Parameters:
    - signal_strength: Strength of the signal from Ink v14 (0 to 1)
    
    Returns:
    - Boolean indicating whether to use Ink v14 (True) or Ink v5 (False)
    """
    return signal_strength >= SIGNAL_STRENGTH_THRESHOLD

## Implement Strategy Switching Logic

Now we'll implement the logic to switch between strategies based on the signal strength from Ink v14.

In [ ]:
def hybrid_ink_strategy(prices, volumes, v14_window=14, v5_window=5):
    """
    Hybrid strategy that switches between Ink v14 and Ink v5 based on signal strength
    
    Parameters:
    - prices: Series or list of historical prices
    - volumes: Series or list of historical volumes
    - v14_window: Window parameter for Ink v14 strategy
    - v5_window: Window parameter for Ink v5 strategy
    
    Returns:
    - final_signal: The trading signal from the selected strategy
    - selected_strategy: Name of the strategy used ('Ink v14' or 'Ink v5')
    - signal_strength: Strength of the Ink v14 signal
    """
    # Get signal and strength from Ink v14
    v14_signal, v14_strength = ink_v14_strategy(prices, volumes, window=v14_window)
    
    # Decide which strategy to use
    if should_use_ink_v14(v14_strength):
        # Use Ink v14 when its signal is strong
        selected_strategy = 'Ink v14'
        final_signal = v14_signal
    else:
        # Use Ink v5 when Ink v14 signal is weak
        selected_strategy = 'Ink v5'
        final_signal = ink_v5_strategy(prices, window=v5_window)
    
    return final_signal, selected_strategy, v14_strength

## Simulate Strategy Switching

Let's simulate the hybrid strategy using sample data to see how it switches between Ink v14 and Ink v5 strategies.

In [ ]:
# Generate synthetic price and volume data for testing
def generate_sample_data(days=100, volatility=0.02):
    """Generate synthetic price and volume data for testing"""
    np.random.seed(42)  # For reproducibility
    
    # Generate price data with a trend and some volatility
    prices = [100.0]  # Starting price
    for _ in range(days-1):
        # Random daily return with some trend
        daily_return = np.random.normal(0.0005, volatility)
        prices.append(prices[-1] * (1 + daily_return))
    
    # Generate corresponding volume data
    volumes = []
    for price in prices:
        # Volume tends to be higher when price is changing rapidly
        base_volume = 10000
        volume_factor = 1 + abs(np.random.normal(0, 0.3))
        volumes.append(int(base_volume * volume_factor))
    
    # Create a date range
    start_date = datetime(2023, 1, 1)
    dates = [start_date + timedelta(days=i) for i in range(days)]
    
    # Return as DataFrame
    return pd.DataFrame({
        'Date': dates,
        'Price': prices,
        'Volume': volumes
    })

# Generate sample data
sample_data = generate_sample_data(days=100)

# Apply hybrid strategy on historical data
results = []
window_size = 20  # For trailing data

for i in range(window_size, len(sample_data)):
    # Get trailing data
    trailing_prices = sample_data['Price'].iloc[i-window_size:i].values
    trailing_volumes = sample_data['Volume'].iloc[i-window_size:i].values
    
    # Apply hybrid strategy
    signal, strategy_used, strength = hybrid_ink_strategy(
        trailing_prices, trailing_volumes, v14_window=14, v5_window=5
    )
    
    # Store results
    results.append({
        'Date': sample_data['Date'].iloc[i],
        'Price': sample_data['Price'].iloc[i],
        'Signal': signal,
        'Strategy': strategy_used,
        'Signal_Strength': strength
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)
results_df.head()

In [ ]:
# Analyze the strategy switching
strategy_counts = results_df['Strategy'].value_counts()
print(f"Strategy usage summary:\n{strategy_counts}")

# Calculate how often we switch strategies
strategy_switches = (results_df['Strategy'] != results_df['Strategy'].shift(1)).sum()
print(f"\nNumber of strategy switches: {strategy_switches}")
print(f"Percentage of time strategy switched: {strategy_switches / len(results_df) * 100:.2f}%")

# Calculate average signal strength when each strategy was used
avg_strength_v14 = results_df[results_df['Strategy'] == 'Ink v14']['Signal_Strength'].mean()
avg_strength_v5 = results_df[results_df['Strategy'] == 'Ink v5']['Signal_Strength'].mean()

print(f"\nAverage signal strength when using Ink v14: {avg_strength_v14:.4f}")
print(f"Average signal strength when using Ink v5: {avg_strength_v5:.4f}")

In [ ]:
# Visualize the strategy switches over time
plt.figure(figsize=(14, 10))

# Plot price chart
ax1 = plt.subplot(3, 1, 1)
ax1.plot(results_df['Date'], results_df['Price'])
ax1.set_title('Price Chart')
ax1.set_ylabel('Price')

# Plot which strategy was used
ax2 = plt.subplot(3, 1, 2, sharex=ax1)
ax2.plot(results_df['Date'], [1 if s == 'Ink v14' else 0 for s in results_df['Strategy']], 
         drawstyle='steps-post')
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Ink v5', 'Ink v14'])
ax2.set_title('Strategy Used')
ax2.grid(True)

# Plot signal strength
ax3 = plt.subplot(3, 1, 3, sharex=ax1)
ax3.plot(results_df['Date'], results_df['Signal_Strength'])
ax3.axhline(y=SIGNAL_STRENGTH_THRESHOLD, color='r', linestyle='--', label=f'Threshold ({SIGNAL_STRENGTH_THRESHOLD})')
ax3.set_title('Ink v14 Signal Strength')
ax3.set_ylabel('Strength')
ax3.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visualize trading signals
plt.figure(figsize=(14, 6))

# Plot price
plt.plot(results_df['Date'], results_df['Price'], label='Price')

# Plot buy signals
buy_points = results_df[results_df['Signal'] == 1]
plt.scatter(buy_points['Date'], buy_points['Price'], marker='^', color='green', s=100, label='Buy')

# Plot sell signals
sell_points = results_df[results_df['Signal'] == -1]
plt.scatter(sell_points['Date'], sell_points['Price'], marker='v', color='red', s=100, label='Sell')

plt.title('Trading Signals from Hybrid Ink Strategy')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.tight_layout()
plt.show()

## Conclusion

This hybrid approach leverages the best of both strategies:

1. When Ink v14 produces strong signals (strength ≥ threshold), we use its recommendations
2. When Ink v14 produces weak signals (strength < threshold), we fall back to Ink v5

This approach can potentially improve overall performance by using the more complex Ink v14 strategy only when it has high conviction, and using the simpler, more robust Ink v5 strategy when Ink v14's signals are less clear.

For a real implementation, you would need to:
1. Replace the simplified strategy functions with your actual Ink v14 and Ink v5 implementations
2. Fine-tune the signal strength threshold based on backtesting 
3. Consider additional factors like market regime, volatility, or time of day that might influence which strategy is more appropriate